## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [15]:
config={"configurable":{"thread_id" : "test-1"}}

In [16]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages:  {response}")
    print(f"Messages: {len(response['messages'])}")

Messages:  {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='772affd2-2eb1-4db2-bf6a-74ee4d3b8c3e'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Task:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy:** 2+2 is universally 4 in standard base-10 arithmetic. No tricks or edge cases apparent.\n6.  **Output Generation:** "2 + 2 equals 4." (or simply "4")\n\nI\'ll keep it direct and accurate.✅\n</think>\n\n2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 17, 'total_tokens': 173, 'completion_time': 0.304701769, 'completion_tokens_details': None, 'prompt_time': 0.000982529, 'prompt_tokens_details': None, 'queue_time': 0.05113238, 'total_time': 0.305684298},

## Token size


In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns ling response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens",550),
            keep=("tokens",200)
        ),
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars


In [18]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: -{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: -539 tokens, 4 messages
[HumanMessage(content='find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='4bcfde25-74f6-4861-bd8d-bfb3aa3ac057'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Identify User Intent:** The user wants to find hotels in Paris.\n2.  **Identify Available Tools:** I have a `search_hotels` tool.\n3.  **Check Tool Parameters:**\n    *   Name: `search_hotels`\n    *   Parameters: `city` (required, string)\n4.  **Extract Parameters from User Input:**\n    *   User said "find Hotels in Paris".\n    *   City = "Paris".\n5.  **Construct Tool Call:**\n    *   Function: `search_hotels`\n    *   Arguments: `{"city": "Paris"}`\n6.  **Execute Tool Call.**\n7.  **Process Tool Response:** (Wait for the output).\n8.  **Formulate Final Answer:** Present the hotel results to the user.\n\n*Self-Correction/Refinement:* The prompt asks me to *find* hotels, which implies using the tool. I will call the tool immediately

## Fraction

In [20]:
from sys import int_info
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

groq_model = init_chat_model("groq:qwen/qwen3.6-27b")
groq_model.profile = {"max_input_tokens": 128000}
@tool
def search_hotels(city:str)  ->str:
    """Search hotels."""
    return f"hotels in {city}: Grad Hotel $350, City Inn $180, Budget Stay $75"

agent = create_agent(
    model=groq_model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=groq_model,
            trigger=("fraction",0.005),
            keep=("fraction", 0.002)
        ),
    ],
)

config = {"configurable" : {"thread_id" : "test-1"}}

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response=agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city} : -{tokens} tokens ({fraction: .4%}), {len(response['messages'])} messages")
    print(response['messages'])

Paris : -196 tokens ( 0.1531%), 4 messages
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='3e22e902-2a43-4360-a73b-728889fc4d36'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user said "Hotels in Paris". This is a clear request to search for hotels in a specific city (Paris).\n2.  **Identify Available Tools:** I have a `search_hotels` function that takes a `city` parameter (required, string).\n3.  **Map Input to Tool:** The city is "Paris". I can directly use this as the `city` parameter.\n4.  **Construct Tool Call:** \n   ```json\n   {\n     "name": "search_hotels",\n     "parameters": {\n       "city": "Paris"\n     }\n   }\n   ```\n5.  **Execute Tool Call:** (Simulated/Expected) I will call the function now.\n6.  **Formulate Response:** I\'ll wait for the tool\'s output and then present the results to the user. Since I\'m generating the tool call now, I\'ll outp

## Human in loop middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID : {email_id}"

def send_email_tool(recipient: str, subject:str, body:str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"
    

In [22]:
agent = create_agent(
    model=groq_model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [23]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='9c8d3170-8efe-4b55-a29e-3e8f5c3cbeb6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI need to use the `send_email_tool`.\nThe recipient is 'john@test.com'.\nThe subject is 'Hello'.\nThe body is 'How are you?'.\nI have all the required parameters.\nI will call the tool now.\n", 'tool_calls': [{'id': 'zbax5ehc1', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 374, 'total_tokens': 495, 'completion_time': 0.230568571, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.028302105, 'prompt_tokens_details': None, 'queue_time': 0.048857934, 'total_time': 0.258870676}, 'model_name': 'qwe

In [25]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject 'Hello'.


In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='9c8d3170-8efe-4b55-a29e-3e8f5c3cbeb6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI need to use the `send_email_tool`.\nThe recipient is 'john@test.com'.\nThe subject is 'Hello'.\nThe body is 'How are you?'.\nI have all the required parameters.\nI will call the tool now.\n", 'tool_calls': [{'id': 'zbax5ehc1', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 374, 'total_tokens': 495, 'completion_time': 0.230568571, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.028302105, 'prompt_tokens_details': None, 'queue_time': 0.048857934, 'total_time': 0.258870676}, 'model_name': 'qwe

# Reject


In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=groq_model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [28]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [29]:
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I attempted to send the email, but the action was rejected. Would you like me to try again, or is there anything else I can help you with?


In [30]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='bfb3f324-b045-4dc6-93a0-7fe17ce751b4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI need to use the `send_email_tool` function.\nThe required parameters are `recipient`, `subject`, and `body`.\nThe user provided:\n- recipient: john@test.com\n- subject: Hello\n- body: How are you?\n\nI have all the necessary information to call the function.\nI will construct the tool call now.\n', 'tool_calls': [{'id': 'tb9rwr1zm', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 374, 'total_tokens': 516, 'completion_time': 0.29709612, 'completion_tokens_details': {'reasoning_tokens': 83}, 'prompt_time': 0.02827776

## Editing

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=groq_model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [32]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [33]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='045e9fee-ef45-4123-84a4-bde3d4265bb8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Analyze the user\'s input: "Send email to wrong@email.com with subject \'Test\' and body \'Hello\'".\n2.  Identify the intent: Send an email.\n3.  Identify the required parameters for `send_email_tool`:\n   - recipient: "wrong@email.com"\n   - subject: "Test"\n   - body: "Hello"\n4.  Check available tools: `send_email_tool` matches the intent.\n5.  Call the tool with the extracted parameters.\n6.  Formulate the response based on the tool call. (No need to wait for output if it\'s a mock, just execute).\n   - `recipient`: "wrong@email.com"\n   - `subject`: "Test"\n   - `body`: "Hello"\n   - All required parameters are present. Proceed. \n   - Tool call: `send_email_tool(recipient="wrong@email.com", subject="Tes

In [34]:
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [35]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='045e9fee-ef45-4123-84a4-bde3d4265bb8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Analyze the user\'s input: "Send email to wrong@email.com with subject \'Test\' and body \'Hello\'".\n2.  Identify the intent: Send an email.\n3.  Identify the required parameters for `send_email_tool`:\n   - recipient: "wrong@email.com"\n   - subject: "Test"\n   - body: "Hello"\n4.  Check available tools: `send_email_tool` matches the intent.\n5.  Call the tool with the extracted parameters.\n6.  Formulate the response based on the tool call. (No need to wait for output if it\'s a mock, just execute).\n   - `recipient`: "wrong@email.com"\n   - `subject`: "Test"\n   - `body`: "Hello"\n   - All required parameters are present. Proceed. \n   - Tool call: `send_email_tool(recipient="wrong@email.com", subject="Tes